# Training Techniques & Hyperparameter Search

**Companion lesson:** https://ml-viz.vercel.app/courses/model-evaluation/03-training-techniques

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor']   = '#1a1d27'
plt.rcParams['text.color']       = 'white'
plt.rcParams['axes.labelcolor']  = '#94a3b8'
plt.rcParams['xtick.color']      = '#94a3b8'
plt.rcParams['ytick.color']      = '#94a3b8'
plt.rcParams['axes.edgecolor']   = '#2e3347'
plt.rcParams['grid.color']       = '#2e3347'

BRAND  = '#818cf8'
TEAL   = '#14b8a6'
YELLOW = '#f59e0b'
ROSE   = '#f43f5e'

rng = np.random.default_rng(42)

## 1. Early Stopping

Training loss decreases monotonically, but validation loss starts rising after ~40 epochs — this is when to stop.

In [ ]:
epochs = np.arange(1, 101)

# Simulated loss curves
train_loss = 1.0 * np.exp(-epochs / 30) + 0.05 + rng.normal(0, 0.005, 100)
val_noise  = rng.normal(0, 0.01, 100)
val_loss   = (1.0 * np.exp(-epochs / 30) + 0.10 +
              0.001 * np.maximum(0, epochs - 40) ** 1.3 + val_noise)

best_epoch = np.argmin(val_loss) + 1

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(epochs, train_loss, color=BRAND,  linewidth=2, label='Train loss')
ax.plot(epochs, val_loss,   color=YELLOW, linewidth=2, label='Validation loss')
ax.axvline(best_epoch, color=TEAL, linestyle='--', linewidth=1.5,
           label=f'Best epoch = {best_epoch} (early stop here)')
ax.fill_between(epochs[best_epoch-1:], train_loss[best_epoch-1:], val_loss[best_epoch-1:],
                alpha=0.12, color=ROSE)
ax.annotate('Overfitting\ngap grows', xy=(75, val_loss[74]),
            xytext=(82, val_loss[74] + 0.05), color=ROSE,
            arrowprops=dict(arrowstyle='->', color=ROSE))
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Early stopping: restore weights from best validation epoch', color='white')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Best epoch: {best_epoch}  val_loss={val_loss[best_epoch-1]:.4f}')
print(f'Final epoch: {len(epochs)}  val_loss={val_loss[-1]:.4f}')
print(f'Generalization gap at final epoch: {val_loss[-1] - train_loss[-1]:.4f}')

## 2. Learning Rate Schedules

Four common schedules over 100 epochs: step decay, cosine annealing, reduce-on-plateau, and warm-up + cosine.

In [ ]:
T = 100
ts = np.arange(T)
lr_max, lr_min = 0.1, 1e-4

# Step decay: reduce by 0.1 every 30 epochs
step_decay = lr_max * (0.1 ** (ts // 30))

# Cosine annealing
cosine = lr_min + 0.5 * (lr_max - lr_min) * (1 + np.cos(np.pi * ts / T))

# Warm-up (10 epochs) then cosine
warmup = 10
warmup_cosine = np.where(
    ts < warmup,
    lr_max * ts / warmup,
    lr_min + 0.5 * (lr_max - lr_min) * (1 + np.cos(np.pi * (ts - warmup) / (T - warmup)))
)

# Reduce on plateau (simulated: reduce by 0.5 at epochs 40, 70)
rop = np.ones(T) * lr_max
for e in [40, 70]:
    rop[e:] *= 0.5

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(ts, step_decay,    color=BRAND,  linewidth=2, label='Step decay (×0.1 every 30)')
ax.plot(ts, cosine,        color=TEAL,   linewidth=2, label='Cosine annealing')
ax.plot(ts, warmup_cosine, color=YELLOW, linewidth=2, label='Warm-up (10) + cosine')
ax.plot(ts, rop,           color=ROSE,   linewidth=2, linestyle='--', label='Reduce on plateau')
ax.axvline(warmup, color=YELLOW, linestyle=':', alpha=0.5)
ax.set_yscale('log')
ax.set_xlabel('Epoch')
ax.set_ylabel('Learning rate (log scale)')
ax.set_title('Learning Rate Schedules', color='white')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Grid Search vs Random Search

For a 2D problem where only the learning rate (x-axis) matters, random search explores far more distinct LR values than a grid.

In [ ]:
# Simulate 25 evaluations for each strategy
n_trials = 25
grid_side = 5  # 5x5 = 25 grid points

# Grid search: 5 LR × 5 batch size values
lr_grid = np.logspace(-5, -1, grid_side)
bs_grid = np.array([16, 64, 128, 256, 512])
grid_lr, grid_bs = np.meshgrid(lr_grid, bs_grid)
grid_lr = grid_lr.ravel()
grid_bs = grid_bs.ravel()

# Random search: 25 independent samples
rand_lr = np.exp(rng.uniform(np.log(1e-5), np.log(1e-1), n_trials))
rand_bs = rng.choice([16, 32, 64, 128, 256, 512], n_trials)

# "True" performance: only LR matters (Gaussian around best LR=1e-3)
def perf(lr): return np.exp(-((np.log10(lr) - np.log10(1e-3))**2) / 0.5)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for ax, lrs, bss, title in [
    (ax1, grid_lr, grid_bs, f'Grid search ({grid_side}×{grid_side}=25 pts)\n{grid_side} unique LR values'),
    (ax2, rand_lr, rand_bs, f'Random search (25 pts)\n{n_trials} unique LR values'),
]:
    perfs = perf(lrs)
    sc = ax.scatter(lrs, bss, c=perfs, cmap='plasma', s=80, vmin=0, vmax=1)
    ax.set_xscale('log')
    ax.set_xlabel('Learning rate')
    ax.set_ylabel('Batch size')
    ax.set_title(title, color='white')
    ax.axvline(1e-3, color=TEAL, linestyle='--', alpha=0.5, label='Optimal LR=1e-3')
    plt.colorbar(sc, ax=ax, label='Performance')
    ax.legend()

plt.suptitle('Grid vs Random Search — only LR matters, batch size is unimportant', color='white')
plt.tight_layout()
plt.show()

print(f'Grid search: {grid_side} unique LR values')
print(f'Random search: {n_trials} unique LR values')
print(f'Best grid performance:   {max(perf(grid_lr)):.4f}')
print(f'Best random performance: {max(perf(rand_lr)):.4f}')

## 4. Bayesian Optimization (Illustrated)

A Gaussian Process surrogate is updated after each evaluation. The acquisition function (Expected Improvement) guides where to sample next — balancing exploration and exploitation.

In [ ]:
# Simulate a 1D Bayesian optimization scenario
# True objective (unknown to optimizer): bimodal with best at x≈0.7
def true_objective(x):
    return -(0.5 * np.exp(-((x - 0.7)**2) / 0.02) +
             0.3 * np.exp(-((x - 0.3)**2) / 0.01)) + 0.1 * rng.standard_normal()

# Initial observations
x_obs = np.array([0.1, 0.4, 0.9])
y_obs = np.array([true_objective(x) for x in x_obs])

# Simple GP posterior (closed-form for RBF kernel, noise=0.01)
def gp_posterior(x_new, x_obs, y_obs, length_scale=0.15, noise=0.05):
    def rbf(a, b): return np.exp(-((a[:, None] - b[None, :])**2) / (2 * length_scale**2))
    K    = rbf(x_obs, x_obs) + noise * np.eye(len(x_obs))
    k_s  = rbf(x_obs, x_new)
    k_ss = rbf(x_new, x_new)
    K_inv  = np.linalg.inv(K)
    mu     = k_s.T @ K_inv @ y_obs
    sigma2 = np.diag(k_ss - k_s.T @ K_inv @ k_s)
    return mu, np.sqrt(np.maximum(sigma2, 0))

x_grid = np.linspace(0, 1, 300)
mu, sigma = gp_posterior(x_grid, x_obs, y_obs)

# Expected Improvement acquisition function
from scipy.stats import norm
f_best = y_obs.min()
z  = (f_best - mu) / (sigma + 1e-9)
ei = sigma * (z * norm.cdf(z) + norm.pdf(z))
next_x = x_grid[np.argmax(ei)]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

# True objective
x_true = np.linspace(0, 1, 300)
y_true_obj = -(0.5 * np.exp(-((x_true - 0.7)**2) / 0.02) +
               0.3 * np.exp(-((x_true - 0.3)**2) / 0.01))
ax1.plot(x_true, y_true_obj, color='gray', linewidth=1, alpha=0.4, label='True objective')
ax1.plot(x_grid, mu, color=BRAND, linewidth=2, label='GP mean')
ax1.fill_between(x_grid, mu - 2*sigma, mu + 2*sigma, alpha=0.15, color=BRAND, label='GP ±2σ')
ax1.scatter(x_obs, y_obs, color=TEAL, s=80, zorder=6, label='Observations')
ax1.axvline(next_x, color=ROSE, linestyle='--', linewidth=1.5, label=f'Next query x={next_x:.2f}')
ax1.set_ylabel('Objective')
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)
ax1.set_title('GP surrogate model after 3 evaluations', color='white')

ax2.plot(x_grid, ei, color=YELLOW, linewidth=2)
ax2.axvline(next_x, color=ROSE, linestyle='--', linewidth=1.5)
ax2.fill_between(x_grid, 0, ei, alpha=0.2, color=YELLOW)
ax2.set_xlabel('Hyperparameter x')
ax2.set_ylabel('Expected Improvement')
ax2.set_title(f'Acquisition function — next query at x={next_x:.2f}', color='white')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## ✏️ Your turn

### Exercise 1 — `early_stopping_epoch(val_losses, patience=5)`

Implement early stopping logic:
- Track the best (lowest) validation loss seen so far
- Count how many consecutive epochs have passed without improvement
- When the count reaches `patience`, stop
- Return the **epoch index** (0-based) of the best validation loss

In [ ]:
def early_stopping_epoch(val_losses, patience=5):
    """
    Find the best epoch using early stopping logic.
    Returns the 0-based index of the best validation loss.
    """
    # TODO(you): iterate over val_losses, track best_loss and best_epoch,
    # increment patience_count when no improvement, stop when patience_count >= patience
    best_loss = float('inf')
    best_epoch = 0
    patience_count = 0
    for i, loss in enumerate(val_losses):
        ...  # fill in
    return best_epoch

In [ ]:
# Test 1: best at index 2, stops after 3 non-improving epochs
losses1 = [0.9, 0.7, 0.5, 0.6, 0.7, 0.8]
result1 = early_stopping_epoch(losses1, patience=3)
assert result1 == 2, f"Expected best_epoch=2, got {result1}"

# Test 2: monotonically decreasing — best is the last epoch
losses2 = [0.9, 0.7, 0.5, 0.3, 0.1]
result2 = early_stopping_epoch(losses2, patience=3)
assert result2 == 4, f"Expected best_epoch=4, got {result2}"

# Test 3: monotonically increasing with patience=2 — best is epoch 0
losses3 = [0.5, 0.6, 0.7, 0.8, 0.9]
result3 = early_stopping_epoch(losses3, patience=2)
assert result3 == 0, f"Expected best_epoch=0, got {result3}"

print(f"Test 1 best_epoch: {result1}  (expected 2)")
print(f"Test 2 best_epoch: {result2}  (expected 4)")
print(f"Test 3 best_epoch: {result3}  (expected 0)")
print("\u2705 Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def early_stopping_epoch(val_losses, patience=5):
    best_loss = float('inf')
    best_epoch = 0
    patience_count = 0
    for i, loss in enumerate(val_losses):
        if loss < best_loss:
            best_loss = loss
            best_epoch = i
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= patience:
                break
    return best_epoch
```

</details>

### Exercise 2 — `random_vs_grid_coverage(n_trials, ...)`

Compare how many **unique learning rate values** grid search vs random search explore.

- **Grid search:** form a `ceil(sqrt(n_trials)) × ceil(sqrt(n_trials))` grid; count distinct LR values
- **Random search:** sample `n_trials` random (lr, bs) pairs from log-uniform LR and categorical BS; count distinct LR values

Return `(n_unique_grid_lr, n_unique_random_lr)`.

In [ ]:
def random_vs_grid_coverage(n_trials, lr_range=(1e-5, 1e-1), bs_range=(16, 512), seed=42):
    """
    Returns (n_unique_grid_lr, n_unique_random_lr).
    """
    rng_c = np.random.default_rng(seed)
    import math

    # TODO(you): build the grid and random samples
    # Grid: side = ceil(sqrt(n_trials)); grid of lr_values × bs_values
    side = math.ceil(math.sqrt(n_trials))
    # lr_values = np.logspace(log10(lr_range[0]), log10(lr_range[1]), side)
    # random: sample n_trials lrs from log-uniform distribution

    n_unique_grid_lr   = ...
    n_unique_random_lr = ...
    return n_unique_grid_lr, n_unique_random_lr

In [ ]:
g, r = random_vs_grid_coverage(n_trials=25)
assert g is not None and r is not None, "returned None"
assert r > g, \
    f"Random search should cover more unique LR values than grid. Got grid={g}, random={r}"
print(f"Grid search unique LR values:   {g}  (should be sqrt(25)=5)")
print(f"Random search unique LR values: {r}  (should be 25)")
print("\u2705 Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def random_vs_grid_coverage(n_trials, lr_range=(1e-5, 1e-1), bs_range=(16, 512), seed=42):
    rng_c = np.random.default_rng(seed)
    import math
    side = math.ceil(math.sqrt(n_trials))

    # Grid: side distinct LR values
    lr_values = np.logspace(np.log10(lr_range[0]), np.log10(lr_range[1]), side)
    n_unique_grid_lr = len(lr_values)  # = side

    # Random: n_trials independent LR samples from log-uniform
    rand_lrs = np.exp(rng_c.uniform(np.log(lr_range[0]), np.log(lr_range[1]), n_trials))
    n_unique_random_lr = len(rand_lrs)  # all unique (continuous distribution)

    return n_unique_grid_lr, n_unique_random_lr
```

</details>